# Lab | Transformers
## Route every message to the right team

Yesterday you built a semantic search for a digital bank's support messages. The bank now wants to go one step further: **route** each new message automatically to the team that handles its intent, before any person reads it.

The support lead has picked the ten intents that agents mix up most often: card delivery, lost and compromised cards, PINs, failed top-ups and exchange rates. If a model can tell these apart, the rest will be easy. Your job is to compare three ways of doing it, and to recommend one:

+ a **zero-shot** model, which needs no labelled examples at all;
+ a **fine-tuned DistilBERT**, trained on the bank's labelled messages;
+ yesterday's **semantic search**, used as a classifier: the intent of the most similar past message.

**The data** is the Banking77 dataset from yesterday's lab, in `data/banking_train.csv` and `data/banking_test.csv`, filtered to the ten intents.

**This lab practises** using and fine-tuning pretrained transformers. You will use, from today's **Transformers and Hugging Face** lesson: `AutoTokenizer`, the zero-shot `pipeline`, `Dataset.map`, `AutoModelForSequenceClassification`, `TrainingArguments` and `Trainer`, and the comparison with the methods of Days 1 and 2.

**By the end, you can:**

1. Apply a zero-shot pipeline and fine-tune DistilBERT to classify customer messages by intent.
2. Evaluate both against a semantic-search baseline and justify which one to use.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on. Fine-tuning takes one to four minutes on a laptop CPU; on a slow computer, open the notebook in [Google Colab](https://colab.research.google.com) with a GPU runtime.

## Setup

You need `transformers`, `torch`, `datasets`, `accelerate` and `sentence-transformers`. If the import fails, install them from a terminal with `pip install transformers torch datasets accelerate sentence-transformers`. The first run downloads three models (about 500 MB in total) unless you already have them from the lesson.

In [1]:
import os
import logging
import warnings
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")

import time
import numpy as np
import pandas as pd
import torch

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
import datasets
datasets.disable_progress_bars()
from sentence_transformers import SentenceTransformer

from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_colwidth", 90)
_ = torch.manual_seed(0)

In [2]:
intents = ["card_arrival", "card_delivery_estimate", "lost_or_stolen_card", "compromised_card", "card_not_working",
           "pin_blocked", "change_pin", "top_up_failed", "top_up_reverted", "exchange_rate"]

train = pd.read_csv("../data/banking_train.csv")
test = pd.read_csv("../data/banking_test.csv")
train = train[train["category"].isin(intents)].reset_index(drop=True)
test = test[test["category"].isin(intents)].reset_index(drop=True)
print(train.shape, test.shape)
train.sample(5, random_state=0)

(1185, 2) (400, 2)


,text,category
833,What do I do if I detect fraudulent use on my car?,compromised_card
567,I lost my card. Can you help me?,lost_or_stolen_card
950,Can my PIN changed remotely without visiting a bank?,change_pin
922,I need to make my card PIN a different number,change_pin
362,how long does it take to get my card i am in a rush,card_delivery_estimate


**Checkpoint:** `(1185, 2) (400, 2)`: 1,185 past messages to learn from, and 400 new messages (40 per intent) to test on.

## Challenge 1: look at the messages as tokens

Before choosing a model, check what the model will actually read.

1. Load the tokenizer of `distilbert/distilbert-base-uncased` and print the tokens of three messages: `"My card hasn't arrived yet"`, `"Someone used my card in Bulgaria!"` and `"Why did my top-up get reverted?"`.
2. Count the tokens of every training message (`len(tokenizer(text)["input_ids"])`) and print the `describe()` of the counts.

In [3]:
checkpoint = "distilbert/distilbert-base-uncased"

# 1. Tokenizer and the tokens of three messages
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

for message in ["My card hasn't arrived yet", "Someone used my card in Bulgaria!", "Why did my top-up get reverted?"]:
    print(f"{message:<35} {tokenizer.tokenize(message)}")

# 2. Number of tokens per training message
token_counts = train["text"].apply(lambda text: len(tokenizer(text)["input_ids"]))   # includes [CLS] and [SEP]
print("\nMessages longer than 64 tokens:", (token_counts > 64).sum(), "\n")
token_counts.describe()

My card hasn't arrived yet          ['my', 'card', 'hasn', "'", 't', 'arrived', 'yet']
Someone used my card in Bulgaria!   ['someone', 'used', 'my', 'card', 'in', 'bulgaria', '!']
Why did my top-up get reverted?     ['why', 'did', 'my', 'top', '-', 'up', 'get', 'reverted', '?']

Messages longer than 64 tokens: 1 



count    1185.000000
mean       15.315612
std         7.985310
min         5.000000
25%        11.000000
50%        13.000000
75%        17.000000
max        66.000000
Name: text, dtype: float64

**Checkpoint:** `hasn't` becomes `hasn`, `'` and `t`, and `top-up` becomes `top`, `-` and `up`. The messages are short: a median of 13 tokens and a mean of about 15, and the longest has 66, including `[CLS]` and `[SEP]`.

**Question:** you will fine-tune with `max_length=64`. How many training messages does that cut, and why is a short `max_length` worth having?

Only **1** of the 1,185 messages is longer than 64 tokens. It has 66 tokens, so it loses just 2 tokens at the end. 75% of the messages have 17 tokens or less, so 64 is plenty.

A short `max_length` is worth having because the trainer pads every batch to its longest message. One very long message would make the whole batch long, and the cost of attention grows with the square of the number of tokens (every token looks at every other token). So a short limit keeps training and prediction faster and uses less memory, and here it costs almost nothing.

## Challenge 2: zero-shot routing

The zero-shot model needs no training, only a description of each intent in plain English. The quality of those descriptions matters: the model compares each message with each description.

1. Complete the dictionary `descriptions` with a short description for the last three intents. Seven are given.
2. Create the zero-shot pipeline with the model `MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33`.
3. Run it on all the test messages, with `candidate_labels=list(descriptions.values())` and `batch_size=16`, and time it.
4. Turn the top label of each result back into an intent and compute the accuracy.

In [4]:
descriptions = {
    "card_arrival": "my new card has not arrived",
    "card_delivery_estimate": "how long card delivery takes",
    "lost_or_stolen_card": "my card was lost or stolen",
    "compromised_card": "someone else used my card",
    "card_not_working": "my card does not work",
    "pin_blocked": "my PIN is blocked",
    "change_pin": "how to change my PIN",
    "top_up_failed": "my top-up did not work",
    "top_up_reverted": "my top-up was reversed",
    "exchange_rate": "which exchange rate is used",
}

# 2. Zero-shot pipeline
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33")

# 3. Predictions for the test messages, timed
start = time.time()
results = zero_shot(test["text"].tolist(), candidate_labels=list(descriptions.values()), batch_size=16)
print(f"{len(test)} messages in {time.time() - start:.0f} seconds")

# 4. Intents and accuracy
description_to_intent = {description: intent for intent, description in descriptions.items()}
zero_shot_pred = [description_to_intent[r["labels"][0]] for r in results]   # labels are sorted by score, so [0] is the top one
zero_shot_accuracy = accuracy_score(test["category"], zero_shot_pred)
print(f"Zero-shot accuracy: {zero_shot_accuracy:.3f}")

400 messages in 4 seconds
Zero-shot accuracy: 0.757


**Checkpoint:** the 400 messages take a few seconds to a minute, and the accuracy is about 0.76 with these descriptions (your own descriptions for the last three intents will move it a few points).

In [5]:
# 5. Which intents does the zero-shot model confuse? Show the crosstab of true and predicted intents.
pd.crosstab(test["category"].to_numpy(), np.array(zero_shot_pred), rownames=["true"], colnames=["predicted"])

predicted,card_arrival,card_delivery_estimate,card_not_working,change_pin,compromised_card,exchange_rate,lost_or_stolen_card,pin_blocked,top_up_failed,top_up_reverted
true,,,,,,,,,,
card_arrival,18,15,0,0,0,0,5,0,0,2
card_delivery_estimate,18,17,1,0,0,2,0,1,0,1
card_not_working,0,0,40,0,0,0,0,0,0,0
change_pin,0,0,0,40,0,0,0,0,0,0
compromised_card,1,0,2,0,26,0,10,1,0,0
exchange_rate,0,0,0,0,0,39,0,0,0,1
lost_or_stolen_card,3,0,3,0,2,0,31,0,0,1
pin_blocked,0,0,6,4,0,0,0,30,0,0
top_up_failed,0,0,2,0,0,0,0,0,36,2


**Question:** which intents does the zero-shot model handle well, and which does it confuse? Why is that hard to fix with better descriptions?

The zero-shot model handles the intents well that have their own words: `change_pin` and `card_not_working` (40/40), `exchange_rate` (39/40) and `top_up_failed` (36/40).

It confuses the intents that talk about the same thing:

+ `card_arrival` and `card_delivery_estimate` are basically a coin flip: 18 vs 15 and 18 vs 17. Both are about a card that is not there yet.
+ `top_up_reverted` goes to `top_up_failed` 13 times.
+ `compromised_card` goes to `lost_or_stolen_card` 10 times.
+ `pin_blocked` goes to `card_not_working` 6 times.

It is hard to fix with better descriptions because the difference between these intents is not in the words. *When will my card get here?* fits "my new card has not arrived" and "how long card delivery takes" equally well. For the customer a reverted top-up is also a top-up that "did not work", and customers rarely use the bank's word "reverted". The model only compares the message with each description, and any description that fits one intent also partly fits its neighbour. A better description moves the errors from one side to the other more than it removes them. The crosstab shows that already: the card mistakes go in both directions. To learn where the bank draws the line between two intents, the model needs labelled examples.

## Challenge 3: fine-tune DistilBERT

Now use the bank's 1,185 labelled messages. Follow the steps of section 6 of the lesson; the skeleton below has the structure, and you fill in the `TODO` parts.

1. Map every intent to a number (`label2id`) and back (`id2label`).
2. Write `to_dataset(df)`: build a `datasets.Dataset` with the columns `text` and `label`, and `map` the tokenizer on it with `truncation=True, max_length=64`.
3. Load `AutoModelForSequenceClassification` with 10 labels, `id2label` and `label2id`.
4. Train for 3 epochs with a batch size of 16 and a learning rate of `5e-5`, evaluating after every epoch.

In [6]:
# 1. Intents as numbers
labels = sorted(intents)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# 2. Datasets
def to_dataset(df):
    data = datasets.Dataset.from_dict({"text": df["text"].tolist(),
                                       "label": [label2id[c] for c in df["category"]]})
    return data.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=64), batched=True)

train_data, test_data = to_dataset(train), to_dataset(test)

# 3. Model with a new classification layer
classifier = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(labels), id2label=id2label, label2id=label2id)

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(prediction.label_ids, predicted)}

# 4. Training
args = TrainingArguments(
    output_dir="intent-classifier",
    num_train_epochs=3,              # passes over the 1,185 messages
    per_device_train_batch_size=16,
    learning_rate=5e-5,              # small: only adjust the pretrained weights
    eval_strategy="epoch",           # score on the test messages after every epoch
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    dataloader_pin_memory=False,
    seed=0,
)
trainer = Trainer(model=classifier, args=args, train_dataset=train_data, eval_dataset=test_data,
                  processing_class=tokenizer, compute_metrics=compute_metrics)

start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} seconds")

{'loss': '1.358', 'grad_norm': '2.814', 'learning_rate': '3.356e-05', 'epoch': '1'}
{'eval_loss': '0.5691', 'eval_accuracy': '0.92', 'eval_runtime': '0.5548', 'eval_samples_per_second': '721', 'eval_steps_per_second': '90.13', 'epoch': '1'}
{'loss': '0.3833', 'grad_norm': '27.89', 'learning_rate': '1.689e-05', 'epoch': '2'}
{'eval_loss': '0.2732', 'eval_accuracy': '0.935', 'eval_runtime': '0.4713', 'eval_samples_per_second': '848.8', 'eval_steps_per_second': '106.1', 'epoch': '2'}
{'loss': '0.1782', 'grad_norm': '6.149', 'learning_rate': '2.222e-07', 'epoch': '3'}
{'eval_loss': '0.2295', 'eval_accuracy': '0.9425', 'eval_runtime': '0.462', 'eval_samples_per_second': '865.8', 'eval_steps_per_second': '108.2', 'epoch': '3'}
{'train_runtime': '36.06', 'train_samples_per_second': '98.58', 'train_steps_per_second': '6.239', 'train_loss': '0.64', 'epoch': '3'}
Training took 36 seconds


In [7]:
# 5. Show the loss, the test loss and the test accuracy of every epoch (trainer.state.log_history)
history = pd.DataFrame(trainer.state.log_history)
history.groupby("epoch").first()[["loss", "eval_loss", "eval_accuracy"]].round(3)

,loss,eval_loss,eval_accuracy
epoch,,,
1.0,1.358,0.569,0.920
2.0,0.383,0.273,0.935
3.0,0.178,0.229,0.942


**Checkpoint:** training takes one to four minutes on a CPU (a few seconds on a GPU). The test accuracy is about 0.92 after the first epoch and about 0.94 after the third, and both the training loss and the test loss fall at every epoch.

**Question:** the classification layer started with random weights. Why does the model still reach a good accuracy after a single pass over 1,185 messages?

Because only the last layer is new. DistilBERT was pretrained on a lot of text, so it already knows the language: messages that mean the same thing already get similar vectors, also when they use different words. The new layer only has to learn a linear mapping from that 768-number vector to the 10 intents, which is basically a logistic regression on top of good embeddings like yesterday. One pass over 1,185 messages is 75 updates with a batch size of 16, and that is enough for this. The pretrained layers only need small adjustments. That's why the test accuracy is already 0.92 after the first epoch, even though the training loss (1.36) is still high.

## Challenge 4: compare with yesterday's semantic search

Yesterday's semantic search can also classify: give a new message the intent of the most similar past message. Use it as the baseline.

1. Encode the training and test messages with `SentenceTransformer("all-MiniLM-L6-v2")` and `normalize_embeddings=True`.
2. For each test message, take the intent of the training message with the highest dot product, and compute the accuracy.
3. Get the predictions of the fine-tuned model on the test set (`trainer.predict(test_data)`, then `argmax` and `id2label`).
4. Build a DataFrame with one row per method (zero-shot, semantic search, fine-tuned DistilBERT), with the accuracy and the number of labelled messages each one used.

In [8]:
# 1. Embeddings
minilm = SentenceTransformer("all-MiniLM-L6-v2")
train_vectors = minilm.encode(train["text"].tolist(), normalize_embeddings=True, batch_size=64)
test_vectors = minilm.encode(test["text"].tolist(), normalize_embeddings=True, batch_size=64)

# 2. Semantic search as a classifier
similarities = test_vectors @ train_vectors.T            # unit vectors, so the dot product is the cosine similarity
search_pred = train["category"].to_numpy()[similarities.argmax(axis=1)]   # intent of the most similar past message
search_accuracy = accuracy_score(test["category"], search_pred)

# 3. Fine-tuned predictions
fine_tuned_pred = [id2label[i] for i in trainer.predict(test_data).predictions.argmax(axis=-1)]
fine_tuned_accuracy = accuracy_score(test["category"], fine_tuned_pred)

# 4. Comparison table
comparison = pd.DataFrame({
    "method": ["zero-shot", "semantic search", "fine-tuned DistilBERT"],
    "labelled messages": [0, len(train), len(train)],
    "accuracy": [zero_shot_accuracy, search_accuracy, fine_tuned_accuracy],
})
comparison.round(3)

Could not fetch README.md from 'sentence-transformers/all-MiniLM-L6-v2' (TypeError: Decompressor.decompress() got an unexpected keyword argument 'output_buffer_limit'). Model will load without a model card.


,method,labelled messages,accuracy
0,zero-shot,0,0.758
1,semantic search,1185,0.915
2,fine-tuned DistilBERT,1185,0.942


**Checkpoint:** zero-shot about 0.76, semantic search about 0.92, fine-tuned DistilBERT about 0.94. Your fine-tuned accuracy can differ by a point or two from run to run and from computer to computer.

In [9]:
# 5. Print the classification report of the fine-tuned model, and the 6 most frequent (true, predicted) mistakes
print(classification_report(test["category"], fine_tuned_pred, digits=3))

errors = pd.DataFrame({"true": test["category"], "predicted": fine_tuned_pred, "text": test["text"]})
errors = errors[errors["true"] != errors["predicted"]]
print(f"{len(errors)} mistakes out of {len(test)}\n")
display(errors.value_counts(["true", "predicted"]).head(6))

# the messages behind the mistakes, to judge whether they are really ambiguous
errors.sort_values(["true", "predicted"])

                        precision    recall  f1-score   support

          card_arrival      0.878     0.900     0.889        40
card_delivery_estimate      0.923     0.900     0.911        40
      card_not_working      0.974     0.950     0.962        40
            change_pin      0.952     1.000     0.976        40
      compromised_card      1.000     0.975     0.987        40
         exchange_rate      1.000     1.000     1.000        40
   lost_or_stolen_card      0.929     0.975     0.951        40
           pin_blocked      0.974     0.925     0.949        40
         top_up_failed      0.848     0.975     0.907        40
       top_up_reverted      0.971     0.825     0.892        40

              accuracy                          0.943       400
             macro avg      0.945     0.942     0.942       400
          weighted avg      0.945     0.943     0.942       400

23 mistakes out of 400



true                    predicted             
top_up_reverted         top_up_failed             7
card_delivery_estimate  card_arrival              4
card_arrival            card_delivery_estimate    3
pin_blocked             change_pin                2
card_arrival            lost_or_stolen_card       1
card_not_working        lost_or_stolen_card       1
Name: count, dtype: int64

,true,predicted,text
5,card_arrival,card_delivery_estimate,When will I get my card?
11,card_arrival,card_delivery_estimate,How long does a card delivery take?
32,card_arrival,card_delivery_estimate,How do I know when my card will arrive?
0,card_arrival,lost_or_stolen_card,How do I locate my card?
81,card_delivery_estimate,card_arrival,my card was not in the mail again can you advise?
84,card_delivery_estimate,card_arrival,I'm just wondering when my card will get here.
111,card_delivery_estimate,card_arrival,When can I expect to receive my new card?
112,card_delivery_estimate,card_arrival,I am waiting for my card to arrive.
127,card_not_working,lost_or_stolen_card,Can you please identify the problem with my bank card?
128,card_not_working,pin_blocked,How do I unblock my card using the app?


**Question:** where does the fine-tuned model still make mistakes? Are they model errors, or messages that a person would also find ambiguous?

23 of the 400 messages are wrong. Almost all of them are between the same pairs of intents that the zero-shot model confused: `top_up_reverted` → `top_up_failed` (7), `card_arrival` ↔ `card_delivery_estimate` (7 in total) and `pin_blocked` → `change_pin` (2). The classification report shows the same: `top_up_reverted` has the lowest recall (0.825) and `card_arrival` the lowest precision (0.878).

Looking at the messages themselves, most of the mistakes are messages a person would also find ambiguous:

+ *When will I get my card?* is labelled `card_arrival`, but *When can I expect to receive my new card?* is labelled `card_delivery_estimate`. That's practically the same question.
+ *What happened to my top-up?*, *How do I reset my PIN?* or *Can I freeze my card right now?* could belong to both intents.
+ In about 7 cases I would have picked the model's intent, not the label. For example *The app wouldn't accept my top up* sounds like a failed top-up, not a reverted one, and *I am waiting for my card to arrive.* sounds like `card_arrival`.

Only about 5 of the 23 look like real model errors: *There appears to have been a reversion in my top up* (the word "reversion" is right there), *...the payment was referted* (probably the typo), *Why didn't my topup go through?*, *Can you please identify the problem with my bank card?* and the police report question that ended up in `card_arrival`. So the model is close to the limit of what these labels allow.

## Challenge 5: route new messages and recommend a method

1. Call `trainer.model.eval()`, then wrap the fine-tuned model in a `text-classification` pipeline (`model=trainer.model, tokenizer=tokenizer`).
2. Route five new messages: four that you write yourself, one for each of four different intents, plus `"I don't know what happened, the app just says error"`. Print each message with its predicted intent and score.

In [10]:
# 1. Pipeline
trainer.model.eval()                     # switch off dropout before predicting
intent_router = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer)

# 2. Five new messages
new_messages = [
    "I ordered my card ten days ago and it's still not in my mailbox",    # card_arrival
    "I entered the wrong PIN three times and now it's locked",            # pin_blocked
    "There's a payment from a shop I've never been to",                    # compromised_card
    "Why did I get less euros than I expected when I paid in dollars?",   # exchange_rate
    "I don't know what happened, the app just says error",                 # none of the ten
]
for message, result in zip(new_messages, intent_router(new_messages)):
    print(f"{result['label']:<24} {result['score']:.2f}   {message}")

card_arrival             0.92   I ordered my card ten days ago and it's still not in my mailbox
pin_blocked              0.97   I entered the wrong PIN three times and now it's locked
compromised_card         0.20   There's a payment from a shop I've never been to
exchange_rate            0.51   Why did I get less euros than I expected when I paid in dollars?
card_not_working         0.66   I don't know what happened, the app just says error


**Checkpoint:** the messages that clearly belong to one intent get the right one with a high score (0.9 or more for the card that has not arrived and for the blocked PIN). Messages that are vague, or that sit between intents, get a lower score: in the reference solution, the unknown payment in a shop goes to `card_not_working` with a score of only 0.26, and the app error message, which belongs to none of the ten intents, is still forced into one of them.

**Question:** the bank asks you which method to put in production. Write a short recommendation (4 to 6 sentences): which method, why, and what should happen to messages like the last one.

I would put the **fine-tuned DistilBERT** in production. It has the highest accuracy (0.942 vs 0.915 for semantic search and 0.758 for zero-shot), it trains in about 40 seconds on a laptop, and it runs on the bank's own machines, so no customer message leaves the bank. Most of its remaining mistakes are between intents that a person can't tell apart either.The problem is messages like the last one: the model always picks one of its ten intents, here `card_not_working` with a score of 0.66. A threshold on the score alone does not solve that, because the correct shop payment message only got 0.20. So I would add an "other" class trained with messages from the remaining Banking77 intents, and send every message with a low score or "other" to a person, and use those messages to label more data.

## Extra: how many labels do you need?

Labelling messages costs the bank time. Find out how the fine-tuned model and the semantic search behave with fewer labels.

1. Write a function that keeps `n` random training messages per intent (`train.groupby("category").sample(n, random_state=0)`).
2. For `n` in 10 and 40, fine-tune a fresh DistilBERT on that subset (same settings as Challenge 3, but 6 epochs for `n=10` because there are so few batches) and compute the test accuracy. Run the semantic search with the same subset as the past messages.
3. Show the results in a table, and add the numbers of Challenge 4 for the full training set.

In [11]:
# 1. n random training messages per intent
def sample_per_intent(n):
    return train.groupby("category").sample(n, random_state=0).reset_index(drop=True)

# 2a. a fresh DistilBERT, same settings as Challenge 3, trained on the subset
def fine_tuned_accuracy_with(subset, epochs):
    torch.manual_seed(0)                 # same start for the new classification layer every time
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(labels), id2label=id2label, label2id=label2id)
    subset_args = TrainingArguments(
        output_dir="intent-classifier-subset",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        learning_rate=5e-5,
        eval_strategy="no",              # only the final accuracy is needed
        logging_strategy="no",
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        dataloader_pin_memory=False,
        seed=0,
    )
    subset_trainer = Trainer(model=model, args=subset_args, train_dataset=to_dataset(subset), processing_class=tokenizer)
    subset_trainer.train()
    predicted = subset_trainer.predict(test_data).predictions.argmax(axis=-1)
    return accuracy_score(test_data["label"], predicted)

# 2b. semantic search with only the subset as past messages
def search_accuracy_with(subset):
    subset_vectors = minilm.encode(subset["text"].tolist(), normalize_embeddings=True, batch_size=64)
    nearest = (test_vectors @ subset_vectors.T).argmax(axis=1)
    return accuracy_score(test["category"], subset["category"].to_numpy()[nearest])

# 3. Results table, plus the full training set from Challenge 4
rows = []
for n, epochs in [(10, 6), (40, 3)]:
    subset = sample_per_intent(n)
    start = time.time()
    rows.append({"labelled per intent": n, "labelled messages": len(subset),
                 "semantic search": search_accuracy_with(subset),
                 "fine-tuned DistilBERT": fine_tuned_accuracy_with(subset, epochs)})
    print(f"n={n}: {time.time() - start:.0f} seconds")

rows.append({"labelled per intent": "all (~119)", "labelled messages": len(train),
             "semantic search": search_accuracy, "fine-tuned DistilBERT": fine_tuned_accuracy})
pd.DataFrame(rows).round(3)

{'train_runtime': '6.119', 'train_samples_per_second': '98.05', 'train_steps_per_second': '6.864', 'train_loss': '1.797', 'epoch': '6'}
n=10: 7 seconds
{'train_runtime': '11.66', 'train_samples_per_second': '103', 'train_steps_per_second': '6.434', 'train_loss': '1.484', 'epoch': '3'}
n=40: 13 seconds


,labelled per intent,labelled messages,semantic search,fine-tuned DistilBERT
0,10,100,0.858,0.685
1,40,400,0.912,0.830
2,all (~119),1185,0.915,0.942


**Conclusions:** With few labels the **semantic search wins clearly**: with 10 messages per intent it gets 0.858 against 0.685 for fine-tuning, and with 40 per intent 0.912 against 0.830. Fine-tuning only beats it with the whole training set (~119 per intent): 0.942 vs 0.915. The two curves cross somewhere between 40 and ~119 labelled messages per intent. The exact point wasn't measured.

It is also interesting how differently the two methods grow. The semantic search is already almost at its maximum with 40 per intent (0.912 vs 0.915), so more labels barely help it. Fine-tuning gains 15 points from 10 to 40 per intent and another 11 points to the full set, and it probably isn't done yet. With 40 per intent and 3 epochs the model only got 75 updates and its training loss was still 1.48, so it was probably undertrained.

So for the bank: start with semantic search while there are only a few dozen labelled messages per intent, it costs nothing to train and a new message can be added any time. Switch to fine-tuning once there are around 100 labelled messages per intent. Limitation: every number comes from one random sample and one seed, so they can move a point or two (more for the small subsets).